In [1]:
pip install mlxtend

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

order_products = pd.read_csv("../data/raw/order_products__prior.csv")
products = pd.read_csv("../data/raw/products.csv")

df = order_products.merge(products, on="product_id")

df = df[['order_id', 'product_name']]

df.head()

,order_id,product_name
0,2,Organic Egg Whites
1,2,Michigan Organic Kale
2,2,Garlic Powder
3,2,Coconut Butter
4,2,Natural Sweetener


In [3]:
unique_orders = df['order_id'].unique()

sample_orders = pd.Series(unique_orders).sample(n=100000, random_state=42)

In [4]:
df_sample = df[df['order_id'].isin(sample_orders)]
transactions = (
    df_sample.groupby('order_id')['product_name']
    .apply(list)
    .tolist()
)

In [5]:
print("Number of baskets:", len(transactions))

import numpy as np
basket_sizes = [len(b) for b in transactions]

print("Average basket size:", np.mean(basket_sizes))
print("Example basket:", transactions[0])

Number of baskets: 100000
Average basket size: 10.09007
Example basket: ['Hair Bender Whole Bean Coffee', 'Organic Whole Milk', 'Organic Mini Homestyle Waffles', 'Naturals Chicken Nuggets', 'Unprocessed American Singles Colby-Style Cheese', 'Organic Mesa Sunrise Cereal', 'Total Greek Strained Yogurt', 'Corn Meal Pizza Crust', 'Sriracha Chili Sauce', 'Organic Broccoli Florets', 'Honeycrisp Apple']


In [7]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

In [8]:
basket_df = pd.DataFrame(te_array, columns=te.columns_)

In [9]:
print(basket_df.shape)
basket_df.head()

(100000, 35118)


,#2 Coffee Filters,#4 Natural Brown Coffee Filters,& Go! Hazelnut Spread + Pretzel Sticks,(70% Juice!) Mountain Raspberry Juice Squeeze,0 Calorie Acai Raspberry Water Beverage,0 Calorie Fuji Apple Pear Water Beverage,0 Calorie Strawberry Dragonfruit Water Beverage,0% Fat Black Cherry Greek Yogurt y,0% Fat Blueberry Greek Yogurt,0% Fat Free Organic Milk,...,with Pump Rebalancing Shampoo,with Seasoned Roasted Potatoes Scrambled Eggs & Sausage,with Sweet Cinnamon Bunches Cereal,with Twist Ties Sandwich & Storage Bags,with Xylitol Cinnamon 18 Sticks Sugar Free Gum,with Xylitol Island Berry Lime 18 Sticks Sugar Free Gum,with Xylitol Original Flavor 18 Sticks Sugar Free Gum,with Xylitol Unwrapped Spearmint 50 Sticks Sugar Free Gum,with Xylitol Watermelon Twist 18 Sticks Sugar Free Gum,with a Splash of Pineapple Coconut Water
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [10]:
item_frequencies = basket_df.sum().sort_values(ascending=False)
item_frequencies

Banana                                                           14469
Bag of Organic Bananas                                           12050
Organic Strawberries                                              8292
Organic Baby Spinach                                              7455
Organic Hass Avocado                                              6681
                                                                 ...  
with Bleach Disinfectant Cleanser Scratch Free Lavender Fresh        1
G Series Perform Orange Flavor Sports Drink                          1
(70% Juice!) Mountain Raspberry Juice Squeeze                        1
with Xylitol Watermelon Twist 18 Sticks Sugar Free Gum               1
with Xylitol Unwrapped Spearmint 50 Sticks Sugar Free Gum            1
Length: 35118, dtype: int64

In [11]:
frequent_items = item_frequencies[item_frequencies >= 100].index
basket_df_filtered = basket_df[frequent_items]

In [12]:
print("Old shape:", basket_df.shape)
print("New shape:", basket_df_filtered.shape)

Old shape: (100000, 35118)
New shape: (100000, 1788)


In [13]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [16]:
frequent_itemsets = fpgrowth(
    basket_df_filtered,
    min_support=0.001,
    use_colnames=True
)

In [17]:
print(frequent_itemsets.shape)
frequent_itemsets.head(10)

(3914, 2)


,support,itemsets
0,0.04283,(Organic Whole Milk)
1,0.02536,(Honeycrisp Apple)
2,0.01043,(Organic Broccoli Florets)
3,0.00836,(Total Greek Strained Yogurt)
4,0.00323,(Naturals Chicken Nuggets)
5,0.00168,(Organic Mini Homestyle Waffles)
6,0.00137,(Sriracha Chili Sauce)
7,0.12050,(Bag of Organic Bananas)
8,0.01065,(Pure Irish Butter)
9,0.00113,(Unsweetened Vanilla Almond Breeze)


In [18]:
print("Number of frequent itemsets:", len(frequent_itemsets))
frequent_itemsets.tail()

Number of frequent itemsets: 3914


,support,itemsets
3909,0.00127,"(Sparkling Lemon Water, Pure Sparkling Water)"
3910,0.00113,"(Bag of Organic Bananas, Pure Sparkling Water)"
3911,0.00106,"(Organic Baby Spinach, Organic Spaghetti Squash)"
3912,0.00120,"(Large Lemon, Shallot)"
3913,0.00114,"(Banana, Shallot)"


In [19]:
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets.head()

,support,itemsets,length
0,0.04283,(Organic Whole Milk),1
1,0.02536,(Honeycrisp Apple),1
2,0.01043,(Organic Broccoli Florets),1
3,0.00836,(Total Greek Strained Yogurt),1
4,0.00323,(Naturals Chicken Nuggets),1


In [26]:
frequent_itemsets[
    frequent_itemsets['length'].isin([2,3])
].sort_values(by='support', ascending=False).head(20)

,support,itemsets,length
2614,0.02013,"(Bag of Organic Bananas, Organic Hass Avocado)",2
1956,0.01899,"(Bag of Organic Bananas, Organic Strawberries)",2
1957,0.01706,"(Organic Strawberries, Banana)",2
2475,0.01565,"(Organic Baby Spinach, Banana)",2
2474,0.01546,"(Bag of Organic Bananas, Organic Baby Spinach)",2
2332,0.01544,"(Banana, Organic Avocado)",2
1856,0.01301,"(Bag of Organic Bananas, Organic Raspberries)",2
2615,0.01278,"(Organic Strawberries, Organic Hass Avocado)",2
2001,0.01242,"(Banana, Strawberries)",2
1981,0.01229,"(Large Lemon, Banana)",2


In [27]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Organic Whole Milk),(Organic Hass Avocado),0.04283,0.06681,0.00471,0.109970,1.646006,1.0,0.001849,1.048492,0.410030,0.044887,0.046250,0.090234
1,(Organic Hass Avocado),(Organic Whole Milk),0.06681,0.04283,0.00471,0.070498,1.646006,1.0,0.001849,1.029767,0.420567,0.044887,0.028906,0.090234
2,(Bag of Organic Bananas),(Organic Whole Milk),0.12050,0.04283,0.00846,0.070207,1.639212,1.0,0.003299,1.029445,0.443378,0.054626,0.028603,0.133866
3,(Organic Whole Milk),(Bag of Organic Bananas),0.04283,0.12050,0.00846,0.197525,1.639212,1.0,0.003299,1.095984,0.407400,0.054626,0.087578,0.133866
4,(Organic Strawberries),(Organic Whole Milk),0.08292,0.04283,0.00771,0.092981,2.170936,1.0,0.004159,1.055292,0.588138,0.065317,0.052395,0.136498


In [28]:
print("Number of rules:", len(rules))
print(rules.columns.tolist())

Number of rules: 5122
['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski']


In [29]:
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,Organic Whole Milk,Organic Hass Avocado,0.04283,0.06681,0.00471,0.109970,1.646006,1.0,0.001849,1.048492,0.410030,0.044887,0.046250,0.090234
1,Organic Hass Avocado,Organic Whole Milk,0.06681,0.04283,0.00471,0.070498,1.646006,1.0,0.001849,1.029767,0.420567,0.044887,0.028906,0.090234
2,Bag of Organic Bananas,Organic Whole Milk,0.12050,0.04283,0.00846,0.070207,1.639212,1.0,0.003299,1.029445,0.443378,0.054626,0.028603,0.133866
3,Organic Whole Milk,Bag of Organic Bananas,0.04283,0.12050,0.00846,0.197525,1.639212,1.0,0.003299,1.095984,0.407400,0.054626,0.087578,0.133866
4,Organic Strawberries,Organic Whole Milk,0.08292,0.04283,0.00771,0.092981,2.170936,1.0,0.004159,1.055292,0.588138,0.065317,0.052395,0.136498


In [30]:
rules_sorted_lift = rules.sort_values(by='lift', ascending=False)
rules_sorted_lift[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

,antecedents,consequents,support,confidence,lift
4254,"Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry ...",Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00103,0.616766,95.327120
4259,Icelandic Style Skyr Blueberry Non-fat Yogurt,"Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry ...",0.00103,0.159196,95.327120
5096,Lemon Sparkling Water,Grapefruit Sparkling Water,0.00104,0.374101,86.000165
5097,Grapefruit Sparkling Water,Lemon Sparkling Water,0.00104,0.239080,86.000165
4255,"Vanilla Skyr Nonfat Yogurt, Icelandic Style Sk...",Non Fat Raspberry Yogurt,0.00103,0.449782,82.076945
4258,Non Fat Raspberry Yogurt,"Vanilla Skyr Nonfat Yogurt, Icelandic Style Sk...",0.00103,0.187956,82.076945
5054,Total 2% Lowfat Greek Strained Yogurt With Blu...,Total 2% with Strawberry Lowfat Greek Strained...,0.00129,0.202830,79.230542
5051,Total 2% with Strawberry Lowfat Greek Strained...,Total 2% Lowfat Greek Strained Yogurt With Blu...,0.00129,0.503906,79.230542
4917,Nonfat Icelandic Style Strawberry Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00168,0.477273,73.767037
4916,Icelandic Style Skyr Blueberry Non-fat Yogurt,Nonfat Icelandic Style Strawberry Yogurt,0.00168,0.259660,73.767037


In [36]:
strong_rules = rules[
    (rules['confidence'] >= 0.10) &
    (rules['lift'] >= 1.5) &
    (rules['support'] >= 0.001)
].copy()

strong_rules = strong_rules.sort_values(by=['lift', 'confidence'], ascending=False)

print("Number of strong rules:", len(strong_rules))
strong_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

Number of strong rules: 1630


,antecedents,consequents,support,confidence,lift
4254,"Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry ...",Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00103,0.616766,95.327120
4259,Icelandic Style Skyr Blueberry Non-fat Yogurt,"Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry ...",0.00103,0.159196,95.327120
5096,Lemon Sparkling Water,Grapefruit Sparkling Water,0.00104,0.374101,86.000165
5097,Grapefruit Sparkling Water,Lemon Sparkling Water,0.00104,0.239080,86.000165
4255,"Vanilla Skyr Nonfat Yogurt, Icelandic Style Sk...",Non Fat Raspberry Yogurt,0.00103,0.449782,82.076945
4258,Non Fat Raspberry Yogurt,"Vanilla Skyr Nonfat Yogurt, Icelandic Style Sk...",0.00103,0.187956,82.076945
5054,Total 2% Lowfat Greek Strained Yogurt With Blu...,Total 2% with Strawberry Lowfat Greek Strained...,0.00129,0.202830,79.230542
5051,Total 2% with Strawberry Lowfat Greek Strained...,Total 2% Lowfat Greek Strained Yogurt With Blu...,0.00129,0.503906,79.230542
4917,Nonfat Icelandic Style Strawberry Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00168,0.477273,73.767037
4916,Icelandic Style Skyr Blueberry Non-fat Yogurt,Nonfat Icelandic Style Strawberry Yogurt,0.00168,0.259660,73.767037


In [37]:
strong_rules['antecedent_len'] = strong_rules['antecedents'].apply(lambda x: len(x.split(', ')))
strong_rules['consequent_len'] = strong_rules['consequents'].apply(lambda x: len(x.split(', ')))

strong_rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len
4254,"Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry ...",Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00167,0.00647,0.00103,0.616766,95.327120,1.0,0.001019,2.592492,0.991165,0.144866,0.614271,0.387981,2,1
4259,Icelandic Style Skyr Blueberry Non-fat Yogurt,"Vanilla Skyr Nonfat Yogurt, Non Fat Raspberry ...",0.00647,0.00167,0.00103,0.159196,95.327120,1.0,0.001019,1.187352,0.995954,0.144866,0.157790,0.387981,1,2
5096,Lemon Sparkling Water,Grapefruit Sparkling Water,0.00278,0.00435,0.00104,0.374101,86.000165,1.0,0.001028,1.590751,0.991127,0.170772,0.371366,0.306591,1,1
5097,Grapefruit Sparkling Water,Lemon Sparkling Water,0.00435,0.00278,0.00104,0.239080,86.000165,1.0,0.001028,1.310546,0.992690,0.170772,0.236959,0.306591,1,1
4255,"Vanilla Skyr Nonfat Yogurt, Icelandic Style Sk...",Non Fat Raspberry Yogurt,0.00229,0.00548,0.00103,0.449782,82.076945,1.0,0.001017,1.807501,0.990084,0.152819,0.446750,0.318869,2,1


In [54]:
simple_rules = strong_rules[
    (strong_rules['antecedent_len'] == 1) &
    (strong_rules['consequent_len'] == 1)
].copy()

simple_rules = simple_rules.sort_values(by=['lift', 'confidence'], ascending=False)

print("Number of simple 1-to-1 rules:", len(simple_rules))
simple_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20)

Number of simple 1-to-1 rules: 1048


,antecedents,consequents,support,confidence,lift
5096,Lemon Sparkling Water,Grapefruit Sparkling Water,0.00104,0.374101,86.000165
5097,Grapefruit Sparkling Water,Lemon Sparkling Water,0.00104,0.239080,86.000165
4917,Nonfat Icelandic Style Strawberry Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00168,0.477273,73.767037
4916,Icelandic Style Skyr Blueberry Non-fat Yogurt,Nonfat Icelandic Style Strawberry Yogurt,0.00168,0.259660,73.767037
4248,Non Fat Raspberry Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00261,0.476277,73.613195
4249,Icelandic Style Skyr Blueberry Non-fat Yogurt,Non Fat Raspberry Yogurt,0.00261,0.403400,73.613195
5099,Non Fat Acai & Mixed Berries Yogurt,Non Fat Raspberry Yogurt,0.00108,0.398524,72.723355
5098,Non Fat Raspberry Yogurt,Non Fat Acai & Mixed Berries Yogurt,0.00108,0.197080,72.723355
5101,Icelandic Style Skyr Blueberry Non-fat Yogurt,Non Fat Acai & Mixed Berries Yogurt,0.00122,0.188563,69.580294
5100,Non Fat Acai & Mixed Berries Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.00122,0.450185,69.580294


In [55]:
simple_rules = simple_rules[
    (simple_rules['lift'] >= 1.5) &
    (simple_rules['lift'] <= 50)
]

In [56]:
simple_rules.sort_values(by='confidence', ascending=False)[
    ['antecedents', 'consequents', 'support', 'confidence', 'lift']
].head(20)

,antecedents,consequents,support,confidence,lift
4831,Zero Calorie Cola,Soda,0.00129,0.472527,41.053647
4818,Organic Yellow Squash,Organic Zucchini,0.00170,0.470914,14.762198
5043,Total 2% Lowfat Greek Strained Yogurt with Peach,Total 2% with Strawberry Lowfat Greek Strained...,0.00256,0.416938,44.402355
431,Organic Fuji Apple,Banana,0.01027,0.369026,2.550461
4688,Gala Apples,Banana,0.00277,0.363041,2.509093
4957,Organic Yellow Peaches,Organic Strawberries,0.00245,0.359765,4.338701
3363,Bartlett Pears,Banana,0.00407,0.359541,2.484903
87,Honeycrisp Apple,Banana,0.00901,0.355284,2.455484
4872,Asparation/Broccolini/Baby Broccoli,Banana,0.00181,0.352827,2.438500
4800,Mango Chunks,Banana,0.00191,0.352399,2.435542


In [58]:
rules.to_csv("../data/processed/all_rules.csv", index=False)
strong_rules.to_csv("../data/processed/strong_rules.csv", index=False)
simple_rules.to_csv("../data/processed/simple_rules.csv", index=False)

print("Rules saved successfully.")

Rules saved successfully.


In [57]:
print("Frequent itemsets:", len(frequent_itemsets))
print("All rules:", len(rules))
print("Strong rules:", len(strong_rules))
print("Simple rules:", len(simple_rules))

Frequent itemsets: 3914
All rules: 5122
Strong rules: 1630
Simple rules: 1026


In [45]:
simple_rules[['antecedents', 'consequents', 'confidence', 'lift']].head(5)

,antecedents,consequents,confidence,lift
5096,Lemon Sparkling Water,Grapefruit Sparkling Water,0.374101,86.000165
5097,Grapefruit Sparkling Water,Lemon Sparkling Water,0.239080,86.000165
4917,Nonfat Icelandic Style Strawberry Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.477273,73.767037
4916,Icelandic Style Skyr Blueberry Non-fat Yogurt,Nonfat Icelandic Style Strawberry Yogurt,0.259660,73.767037
4248,Non Fat Raspberry Yogurt,Icelandic Style Skyr Blueberry Non-fat Yogurt,0.476277,73.613195
